# Notebook 1: Fetch NCBI SRA Stats for All PlasticDB Organisms

Queries the NCBI Sequence Read Archive (SRA) via the public E-utilities JSON API for every unique organism in PlasticDB.

For each organism the following real values are retrieved:
- Total number of SRA runs deposited under that organism name
- Sequencing platforms used (Illumina, PacBio, Nanopore, etc.)
- Library strategies (WGS, AMPLICON, RNA-Seq, etc.)
- Total sequenced bases (summed across up to 200 sampled runs)
- Deposit year range

Results are cached to `../data/sra_stats.csv`. Re-running this notebook skips organisms already in the cache.

Rate: 2.5 requests per second (NCBI guideline without API key). Estimated runtime for 944 organisms: approximately 7 minutes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import re
import time
import requests
import pandas as pd

PDB_TSV  = Path.cwd().parent.parent / 'plastic-biodegradation-analysis' / 'data' / 'plasticdb_microorganisms.tsv'
SRA_CSV  = Path.cwd().parent / 'data' / 'sra_stats.csv'
SRA_CSV.parent.mkdir(exist_ok=True)

NCBI_BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
session = requests.Session()
session.headers['User-Agent'] = 'igem-toronto-research/1.0'

print('Paths set.')

In [ ]:
# Load PlasticDB and extract unique organisms
df = pd.read_csv(PDB_TSV, sep='\t', dtype=str, on_bad_lines='skip')
df.columns = [
    'organism','tax_id','plastic','reference','enzyme_name',
    'enzyme_id','db_enzyme_name','gene','genbank_id','sequence',
    'year','evidence','plastic_used','manufacturer','analytical_grade',
    'thermophilic','isolation_sample','isolation_environment',
    'isolation_location','extrapolated_from_enzyme','enzyme_id_in_paper','doi',
]
df['organism'] = df['organism'].str.strip()

orgs = (
    df.groupby('organism')
    .agg(tax_id=('tax_id','first'), n_entries=('organism','count'))
    .reset_index()
    .sort_values('n_entries', ascending=False)
)
print(f'Unique organisms: {len(orgs)}')
print(orgs.head(10).to_string(index=False))

In [ ]:
def fetch_sra_one(organism: str, sleep: float = 0.4) -> dict:
    """Fetch SRA stats for one organism via NCBI E-utilities."""
    time.sleep(sleep)
    term = f'"{organism}"[Organism]'
    url  = (f"{NCBI_BASE}/esearch.fcgi"
            f"?db=sra&term={requests.utils.quote(term)}&retmax=200&retmode=json")
    try:
        r = session.get(url, timeout=20)
        r.raise_for_status()
        data  = r.json()['esearchresult']
        count = int(data.get('count', 0))
        ids   = data.get('idlist', [])
    except Exception as e:
        return {'organism': organism, 'sra_run_count': None, 'error': str(e)}

    if count == 0 or not ids:
        return {'organism': organism, 'sra_run_count': 0,
                'sra_platforms': '', 'sra_strategies': '',
                'sra_total_bases': None, 'sra_date_range': ''}

    # Fetch summaries for platform/strategy breakdown
    time.sleep(sleep)
    uid_str = ','.join(ids[:200])
    sumurl  = f"{NCBI_BASE}/esummary.fcgi?db=sra&id={uid_str}&retmode=json"
    try:
        sr = session.get(sumurl, timeout=30)
        sr.raise_for_status()
        result = sr.json().get('result', {})
    except Exception:
        return {'organism': organism, 'sra_run_count': count,
                'sra_platforms': '', 'sra_strategies': '',
                'sra_total_bases': None, 'sra_date_range': ''}

    platforms, strategies, total_bases, dates = set(), set(), 0, []
    for uid, entry in result.items():
        if uid == 'uids':
            continue
        x = entry.get('expxml', '')
        m = re.search(r'<Platform[^>]*>([^<]+)<', x)
        if m:
            platforms.add(m.group(1).strip())
        m = re.search(r'total_bases="(\d+)"', x)
        if m:
            total_bases += int(m.group(1))
        m = re.search(r'<LIBRARY_STRATEGY>([^<]+)</LIBRARY_STRATEGY>', x)
        if m:
            strategies.add(m.group(1).strip())
        cd = entry.get('createdate', '')[:4]
        if cd:
            dates.append(cd)

    return {
        'organism':        organism,
        'sra_run_count':   count,
        'sra_platforms':   '; '.join(sorted(platforms)),
        'sra_strategies':  '; '.join(sorted(strategies)),
        'sra_total_bases': total_bases or None,
        'sra_date_range':  f"{min(dates)}-{max(dates)}" if dates else '',
    }

print('fetch_sra_one() defined.')

In [ ]:
# Test on one organism before running the full batch
test = fetch_sra_one('Ideonella sakaiensis')
print(test)

In [ ]:
# Load cache, skip already-fetched organisms
if SRA_CSV.exists():
    cached = pd.read_csv(SRA_CSV, dtype=str)
    done   = set(cached['organism'].tolist())
    print(f'Cache: {len(cached)} rows already fetched.')
else:
    cached = pd.DataFrame()
    done   = set()

todo = orgs[~orgs['organism'].isin(done)].copy()
print(f'Remaining: {len(todo)} organisms to fetch.')

In [ ]:
# Fetch all remaining organisms
rows = []
for i, (_, row) in enumerate(todo.iterrows(), 1):
    result = fetch_sra_one(row['organism'])
    rows.append(result)
    if i % 50 == 0:
        print(f'  {i}/{len(todo)} done')

if rows:
    new_df   = pd.DataFrame(rows)
    combined = pd.concat([cached, new_df], ignore_index=True) if not cached.empty else new_df
    combined.to_csv(SRA_CSV, index=False)
    print(f'Saved {len(combined)} rows to {SRA_CSV}')
else:
    combined = cached
    print('Nothing new to fetch.')

In [ ]:
# Summary of SRA results
sra = pd.read_csv(SRA_CSV)
sra['sra_run_count'] = pd.to_numeric(sra['sra_run_count'], errors='coerce')

print(f'Total organisms: {len(sra)}')
print(f'With SRA data (runs > 0): {(sra["sra_run_count"] > 0).sum()}')
print(f'With no SRA runs:         {(sra["sra_run_count"] == 0).sum()}')
print(f'Fetch error:              {sra["sra_run_count"].isna().sum()}')
print()
print('Top 15 organisms by SRA run count:')
print(sra.nlargest(15, 'sra_run_count')[['organism','sra_run_count','sra_platforms','sra_strategies']].to_string(index=False))